In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import os
from utility.ffqd_mnist import FFQD_Dataset, label_for_id
from torch.utils.data import DataLoader
DATA_PATH='/leonardo_work/tra26_ictpai/hen_forma_data/ffqd_mnist/'
if not os.path.exists(DATA_PATH):
    DATA_PATH='ffqd_mnist/'

training_data = FFQD_Dataset(DATA_PATH, train=True)
train_dataloader = DataLoader(training_data, batch_size=64)

test_data = FFQD_Dataset(DATA_PATH)
test_dataloader = DataLoader(test_data, batch_size=64)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, input_size = 784, output_size=10):
        super(MLP, self).__init__()
        # self.fc = torch.nn.Linear(input_size,output_size)
        # self.relu = torch.nn.ReLU()
        # self.linear1 = nn.Linear(input_size, 512)
        # self.relu1 = nn.ReLU()
        self.mlp_stack = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, output_size)
        )

    def forward(self, x):
        # y  = self.linear1(x)
        # y = self.relu1(y)
        y = self.mlp_stack(x)
        return y

In [ ]:
model = MLP().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=.01)
criterion = torch.nn.CrossEntropyLoss()

def reset_model():
    global model, optimizer
    model = MLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=.01)


In [ ]:
def train_one_epoch(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch_id, (X,y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # calculate loss
        y_pred = model(X)
        loss = loss_fn(y_pred, y)


        # print( y_pred.dtype, y_pred.shape)
        # print(loss)

        # backprop loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch_id % 100 == 0:
            print(f"({batch_id}) loss: {loss:>7}")


In [ ]:
def test_one_epoch(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X,y in dataloader:
            X, y = X.to(device), y.to(device)
            pred_y = model(X)
            test_loss += loss_fn(pred_y, y).item()
            correct += (pred_y.argmax(1) == y.argmax(1)).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    return test_loss, correct


In [ ]:
reset_model()
# old_weights = torch.Tensor(np.array(model.get_parameter("fc.weight").detach().numpy())) # copy old weights
old_weights = model.get_parameter("mlp_stack.0.weight").clone().detach().cpu()
for i in range(5):
    train_one_epoch(train_dataloader, model, criterion, optimizer)
    print(test_one_epoch(test_dataloader, model, criterion))

In [ ]:
for name, par in model.named_parameters():
    print(name, par.shape)

In [ ]:
l0_weights = model.get_parameter('mlp_stack.0.weight').detach().cpu()
l0_weights_diff = l0_weights- old_weights
f, axs = plt.subplots(10,10,figsize=(24,24))
axs = axs.flatten()
start=300
for i in range(start,start+100):
    axs[i-start].imshow(l0_weights_diff[i].clip(min=0).view(28,28), cmap="gray_r")

### And now ?
<pre>


















</pre>

